# 003 — Cascaded Inference (3 Stages)

```
音訊 + 問診四題（516維）
  ↓
Stage 1: covid vs non-covid
  ├─ covid       → 輸出信心分數
  └─ non-covid
        ↓
     Stage 2: healthy vs symptomatic
        ├─ healthy     → 輸出信心分數
        └─ symptomatic
              ↓
           Stage 3: upper_infection vs lower_infection vs obstructive_disease
              → 輸出三子類信心分數
```

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
import librosa
import tensorflow as tf
from huggingface_hub import from_pretrained_keras
from huggingface_hub.utils import HfFolder

c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 載入模型

In [2]:
print('載入 HeAR...')
hear_model = from_pretrained_keras('google/hear')
serving_fn  = hear_model.signatures['serving_default']
print('HeAR 載入完成')

ckpt_s1 = joblib.load('checkpoints/stage1_covid_vs_noncovid.pkl')
ckpt_s2 = joblib.load('checkpoints/stage2_healthy_vs_symptomatic.pkl')
ckpt_s3 = joblib.load('checkpoints/stage3_symptomatic_subtype.pkl')

# clf_s1, scaler_s1, encoder_s1 = ckpt_s1['clf'], ckpt_s1['scaler'], ckpt_s1['encoder']
# clf_s2, scaler_s2, encoder_s2 = ckpt_s2['clf'], ckpt_s2['scaler'], ckpt_s2['encoder']
# clf_s3, scaler_s3, encoder_s3 = ckpt_s3['clf'], ckpt_s3['scaler'], ckpt_s3['encoder']

# 載入模型時（不再需要 scaler）
clf_s1, encoder_s1 = ckpt_s1["clf"], ckpt_s1["encoder"]
clf_s2, encoder_s2 = ckpt_s2["clf"], ckpt_s2["encoder"]
clf_s3, encoder_s3 = ckpt_s3["clf"], ckpt_s3["encoder"]

FEATURES = ckpt_s1['features']

print('Stage 1 classes:', encoder_s1.classes_)
print('Stage 2 classes:', encoder_s2.classes_)
print('Stage 3 classes:', encoder_s3.classes_)

載入 HeAR...


Fetching 24 files: 100%|██████████| 24/24 [00:00<?, ?it/s]

HeAR 載入完成
Stage 1 classes: ['covid' 'non-covid']
Stage 2 classes: ['healthy' 'symptomatic']
Stage 3 classes: ['lower_infection' 'obstructive_disease' 'upper_infection']


## 選擇推論檔案

In [13]:
COUGHVID_ROOT = 'C:/Users/aint/Desktop/RNN/Final_project/coughvid_wav'
AUG_ROOT      = 'data/augmented_wav'
TEST_CSV      = 'data/prepared_test_split_hear.csv'

df_test = pd.read_csv(TEST_CSV)
sample  = df_test.sample(1).iloc[0]
# sample = df_test.sample(1, random_state=42).iloc[0]

filename = sample['filename']

# 根據檔名決定路徑
if filename.startswith('aug_'):
    fn_wav = os.path.join(AUG_ROOT, filename)
else:   
    fn_wav = os.path.join(COUGHVID_ROOT, filename)

print(f'檔案:     {filename}')
print(f'真實標籤: {sample["label"]}')
print(f'路徑:     {fn_wav}')

檔案:     aug_e6264c42-d888-4a82-97ac-6c5a40c1cc19_stretch_slow.wav
真實標籤: symptomatic
路徑:     data/augmented_wav\aug_e6264c42-d888-4a82-97ac-6c5a40c1cc19_stretch_slow.wav


## 問診四題（模擬）

In [14]:
age                   = float(sample.get('age', 35))
gender_encoded        = int(sample.get('gender_encoded', -1))
respiratory_condition = int(sample.get('respiratory_condition', 0))
fever_muscle_pain     = int(sample.get('fever_muscle_pain', 0))

gender_display = {0: '男', 1: '女', 2: '其他', -1: '未知'}
print(f'問診結果:')
print(f'  年齡:           {age}')
print(f'  性別:           {gender_display.get(gender_encoded, "未知")}')
print(f'  慢性呼吸道疾病: {"是" if respiratory_condition else "否"}')
print(f'  發燒或肌肉痠痛: {"是" if fever_muscle_pain else "否"}')

問診結果:
  年齡:           35.0
  性別:           男
  慢性呼吸道疾病: 否
  發燒或肌肉痠痛: 否


## HeAR Embedding

In [15]:
SAMPLE_RATE  = 16000
CLIP_SAMPLES = SAMPLE_RATE * 2

def get_embedding(file_path):
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    if len(audio) < CLIP_SAMPLES:
        audio = np.pad(audio, (0, CLIP_SAMPLES - len(audio)))
    n_clips = len(audio) // CLIP_SAMPLES
    clips   = np.array([audio[i*CLIP_SAMPLES:(i+1)*CLIP_SAMPLES]
                        for i in range(n_clips)], dtype=np.float32)
    return serving_fn(x=tf.constant(clips))['output_0'].numpy().mean(axis=0)

emb = get_embedding(fn_wav)
X_input = np.concatenate([
    emb, [age, gender_encoded, respiratory_condition, fever_muscle_pain]
]).reshape(1, -1).astype(np.float32)
print(f'X_input.shape: {X_input.shape}')

# 問診時無法取得 dyspnea/wheezing/congestion，設為 -1
# cough_type 如果有問診資料就填入，否則也設 -1

cough_type_encoded = -1   # 可改成從問診答案取得

X3_input = np.concatenate([
    emb,
    [age, gender_encoded, respiratory_condition, fever_muscle_pain,
     cough_type_encoded, -1, -1, -1]   # dyspnea, wheezing, congestion
]).reshape(1, -1).astype(np.float32)

print(f'X3_input.shape: {X3_input.shape}')   # 應為 (1, 520)

X_input.shape: (1, 516)
X3_input.shape: (1, 520)


## Cascaded Inference（三階段）

In [16]:
CONFIDENCE_THRESHOLD = 0.60

def print_bar(label, prob, width=25):
    bar = '█' * int(prob * width)
    print(f'  {label:<25} {prob*100:5.1f}%  {bar}')

# ── Stage 1: covid vs non-covid ──
# prob_s1    = clf_s1.predict_proba(scaler_s1.transform(X_input))[0]
prob_s1 = clf_s1.predict_proba(X_input)[0]
pred_s1    = encoder_s1.classes_[np.argmax(prob_s1)]
p_covid    = prob_s1[list(encoder_s1.classes_).index('covid')]
p_noncovid = 1.0 - p_covid

print('=' * 50)
print('Stage 1: COVID vs Non-COVID')
print_bar('covid',     p_covid)
print_bar('non-covid', p_noncovid)
print(f'  → 判定: {pred_s1}')
print()

if pred_s1 == 'covid':
    print('=' * 50)
    uncertain = p_covid < CONFIDENCE_THRESHOLD
    status = '（信心不足）' if uncertain else ''
    print(f'最終結果: COVID-19 {status}')
    print(f'  信心分數: {p_covid*100:.1f}%')

else:
    # ── Stage 2: healthy vs symptomatic ──
    # prob_s2       = clf_s2.predict_proba(scaler_s2.transform(X_input))[0]
    prob_s2 = clf_s2.predict_proba(X_input)[0]
    pred_s2       = encoder_s2.classes_[np.argmax(prob_s2)]
    p_healthy     = p_noncovid * prob_s2[list(encoder_s2.classes_).index('healthy')]
    p_symptomatic = p_noncovid * prob_s2[list(encoder_s2.classes_).index('symptomatic')]

    print('Stage 2: Healthy vs Symptomatic')
    print_bar('healthy',     p_healthy     / (p_covid + p_healthy + p_symptomatic))
    print_bar('symptomatic', p_symptomatic / (p_covid + p_healthy + p_symptomatic))
    print(f'  → 判定: {pred_s2}')
    print()

    if pred_s2 == 'healthy':
        conf = p_healthy / (p_covid + p_healthy + p_symptomatic)
        print('=' * 50)
        uncertain = conf < CONFIDENCE_THRESHOLD
        status = '（信心不足）' if uncertain else ''
        print(f'最終結果: Healthy {status}')
        print(f'  信心分數: {conf*100:.1f}%')

    else:
        # ── Stage 3: upper / lower / obstructive ──
        # prob_s3  = clf_s3.predict_proba(scaler_s3.transform(X_input))[0]
        prob_s3 = clf_s3.predict_proba(X3_input)[0]
        pred_s3  = encoder_s3.classes_[np.argmax(prob_s3)]

        print('Stage 3: Symptomatic Subtype')
        for cls, prob in zip(encoder_s3.classes_, prob_s3):
            print_bar(cls, prob)
        print(f'  → 判定: {pred_s3}')
        print()

        print('=' * 50)
        conf = prob_s3[np.argmax(prob_s3)]
        uncertain = conf < CONFIDENCE_THRESHOLD
        status = '（信心不足）' if uncertain else ''
        print(f'最終結果: Symptomatic — {pred_s3} {status}')
        print()

        # 最終五類信心分數（正規化）
        p_upper       = p_symptomatic * prob_s3[list(encoder_s3.classes_).index('upper_infection')]
        p_lower       = p_symptomatic * prob_s3[list(encoder_s3.classes_).index('lower_infection')]
        p_obstructive = p_symptomatic * prob_s3[list(encoder_s3.classes_).index('obstructive_disease')]
        total = p_covid + p_healthy + p_upper + p_lower + p_obstructive

        print('各類別信心分數（五類）:')
        for cls, prob in sorted([
            ('covid',               p_covid       / total),
            ('healthy',             p_healthy     / total),
            ('upper_infection',     p_upper       / total),
            ('lower_infection',     p_lower       / total),
            ('obstructive_disease', p_obstructive / total),
        ], key=lambda x: -x[1]):
            print_bar(cls, prob)

print()

# 真實標籤顯示
true_label  = sample['label']
true_expert = sample.get('expert_label', '未知')

if true_label == 'symptomatic':
    print(f'真實標籤: {true_label}（子類：{true_expert}）')
else:
    print(f'真實標籤: {true_label}')

# 判斷正確與否
if pred_s1 == 'covid':
    final_pred = 'covid'
elif pred_s2 == 'healthy':
    final_pred = 'healthy'
else:
    final_pred = pred_s3

label_correct   = (final_pred == true_label) or \
                  (final_pred in ['upper_infection', 'lower_infection', 'obstructive_disease']
                   and true_label == 'symptomatic')
subtype_correct = (final_pred == true_expert)

print(f'結果: {"✓ 正確" if label_correct else "✗ 錯誤"}', end='')
if true_label == 'symptomatic':
    print(f'（子類: {"✓ 正確" if subtype_correct else "✗ 錯誤"}）')
else:
    print()

Stage 1: COVID vs Non-COVID
  covid                       0.2%  
  non-covid                  99.8%  ████████████████████████
  → 判定: non-covid

Stage 2: Healthy vs Symptomatic
  healthy                     0.2%  
  symptomatic                99.6%  ████████████████████████
  → 判定: symptomatic

Stage 3: Symptomatic Subtype
  lower_infection             0.0%  
  obstructive_disease       100.0%  ████████████████████████
  upper_infection             0.0%  
  → 判定: obstructive_disease

最終結果: Symptomatic — obstructive_disease 

各類別信心分數（五類）:
  obstructive_disease        99.6%  ████████████████████████
  healthy                     0.2%  
  covid                       0.2%  
  upper_infection             0.0%  
  lower_infection             0.0%  

真實標籤: symptomatic（子類：obstructive_disease）
結果: ✓ 正確（子類: ✓ 正確）


In [17]:
import joblib
ckpt = joblib.load('checkpoints/stage3_symptomatic_subtype.pkl')

# 看訓練時用的 feature 名稱清單
print(ckpt['features'])
print("總數:", len(ckpt['features']))

['emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'emb_19', 'emb_20', 'emb_21', 'emb_22', 'emb_23', 'emb_24', 'emb_25', 'emb_26', 'emb_27', 'emb_28', 'emb_29', 'emb_30', 'emb_31', 'emb_32', 'emb_33', 'emb_34', 'emb_35', 'emb_36', 'emb_37', 'emb_38', 'emb_39', 'emb_40', 'emb_41', 'emb_42', 'emb_43', 'emb_44', 'emb_45', 'emb_46', 'emb_47', 'emb_48', 'emb_49', 'emb_50', 'emb_51', 'emb_52', 'emb_53', 'emb_54', 'emb_55', 'emb_56', 'emb_57', 'emb_58', 'emb_59', 'emb_60', 'emb_61', 'emb_62', 'emb_63', 'emb_64', 'emb_65', 'emb_66', 'emb_67', 'emb_68', 'emb_69', 'emb_70', 'emb_71', 'emb_72', 'emb_73', 'emb_74', 'emb_75', 'emb_76', 'emb_77', 'emb_78', 'emb_79', 'emb_80', 'emb_81', 'emb_82', 'emb_83', 'emb_84', 'emb_85', 'emb_86', 'emb_87', 'emb_88', 'emb_89', 'emb_90', 'emb_91', 'emb_92', 'emb_93', 'emb_94', 'emb_95', 'emb_96', 'emb_97', 'emb_98', 'emb_99', 'emb_100'